In [11]:
from src.envs import N3il

In [12]:
n = 5 # Example grid size, can be adjusted as needed
i = 0 # Random seed index, can be adjusted as needed

args = {
    'algorithm': 'MCTS',
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 10*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': False,
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': f'tests/tests_mcts',  # Directory to save tables
    'figure_dir': f'tests/tests_mcts/figure',  # Directory to save figures
    'random_seed': i,  # Use the loop index as a seed for reproducibility
}

n3il_test = N3il((n,n), args=args)

In [13]:
state = n3il_test.get_initial_state()
state = n3il_test.get_next_state(state, 0)  # Example move
state

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [14]:
from src.algos.mcts import Node
import numpy as np

class Node_with_symmetry(Node):
    """
    Enhanced Node class with symmetry detection capabilities.
    Inherits all MCTS functionality from Node and adds symmetry analysis.
    """
    
    def __init__(self, game, args, state, parent=None, action_taken=None):
        # Initialize the parent Node class
        super().__init__(game, args, state, parent, action_taken)
        
        # Cache for symmetry results to avoid recomputation
        self._symmetry_cache = None
        self._symmetry_computed = False
    
    def detect_symmetries(self, use_cache=True):
        """
        Detect all symmetry properties of the current state in a single method.
        
        This method checks for:
        - Central symmetry (180-degree rotation)
        - Horizontal symmetry (flip up-down)  
        - Vertical symmetry (flip left-right)
        - Main diagonal symmetry (transpose)
        - Anti-diagonal symmetry (anti-diagonal flip)
        
        Args:
            use_cache (bool): Whether to use cached results if available
            
        Returns:
            dict: Dictionary with symmetry properties as boolean values
        """
        if use_cache and self._symmetry_computed:
            return self._symmetry_cache
        
        # Perform all symmetry checks in one place
        results = {
            'central': np.allclose(self.state, np.flip(self.state, (0, 1))),
            'horizontal': np.allclose(self.state, np.flipud(self.state)),
            'vertical': np.allclose(self.state, np.fliplr(self.state)),
            'main_diagonal': np.allclose(self.state, self.state.T),
            'anti_diagonal': np.allclose(self.state, np.flip(self.state.T, (0, 1))),
        }
        
        # Cache the results
        self._symmetry_cache = results
        self._symmetry_computed = True
        
        return results

In [15]:
# Test the Node_with_symmetry class
print("=== Testing Node_with_symmetry Class ===")
print()

# Create a node with symmetry detection
node_with_sym = Node_with_symmetry(n3il_test, args, state)
node_with_sym.detect_symmetries()

=== Testing Node_with_symmetry Class ===



{'central': False,
 'horizontal': False,
 'vertical': False,
 'main_diagonal': True,
 'anti_diagonal': False}

In [ ]:
import numpy as np
from numba import njit

@njit(cache=True, nogil=True)
def apply_symmetry_transform_nb(action, transform_type, row_count, column_count):
    """
    Apply a symmetry transformation to an action (flattened index).
    
    Args:
        action (int): Flattened action index
        transform_type (int): Type of transformation (0-7)
        row_count (int): Number of rows
        column_count (int): Number of columns
    
    Returns:
        int: Transformed action index
    """
    # Convert action to 2D coordinates
    row = action // column_count
    col = action % column_count
    
    # Apply transformation based on type
    if transform_type == 0:  # Identity
        new_row, new_col = row, col
    elif transform_type == 1:  # Horizontal flip
        new_row, new_col = row_count - 1 - row, col
    elif transform_type == 2:  # Vertical flip
        new_row, new_col = row, column_count - 1 - col
    elif transform_type == 3:  # 180-degree rotation (central symmetry)
        new_row, new_col = row_count - 1 - row, column_count - 1 - col
    elif transform_type == 4:  # Main diagonal (transpose)
        new_row, new_col = col, row
    elif transform_type == 5:  # Anti-diagonal
        new_row, new_col = column_count - 1 - col, row_count - 1 - row
    elif transform_type == 6:  # Horizontal flip + transpose
        new_row, new_col = col, row_count - 1 - row
    elif transform_type == 7:  # Vertical flip + transpose
        new_row, new_col = column_count - 1 - col, row
    else:
        new_row, new_col = row, col  # Default to identity
    
    # Convert back to flattened index
    return new_row * column_count + new_col

@njit(cache=True, nogil=True)
def detect_state_symmetries_nb(state):
    """
    Detect which symmetries the current state possesses.
    
    Args:
        state (np.ndarray): 2D state array
    
    Returns:
        np.ndarray: Boolean array indicating which symmetries are present
                   [horizontal, vertical, central, main_diag, anti_diag]
    """
    row_count, column_count = state.shape
    symmetries = np.zeros(5, dtype=np.bool_)
    
    # Horizontal symmetry (flip up-down)
    symmetries[0] = True
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] != state[row_count - 1 - i, j]:
                symmetries[0] = False
                break
        if not symmetries[0]:
            break
    
    # Vertical symmetry (flip left-right)
    symmetries[1] = True
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] != state[i, column_count - 1 - j]:
                symmetries[1] = False
                break
        if not symmetries[1]:
            break
    
    # Central symmetry (180-degree rotation)
    symmetries[2] = True
    for i in range(row_count):
        for j in range(column_count):
            if state[i, j] != state[row_count - 1 - i, column_count - 1 - j]:
                symmetries[2] = False
                break
        if not symmetries[2]:
            break
    
    # Main diagonal symmetry (transpose) - only for square grids
    if row_count == column_count:
        symmetries[3] = True
        for i in range(row_count):
            for j in range(column_count):
                if state[i, j] != state[j, i]:
                    symmetries[3] = False
                    break
            if not symmetries[3]:
                break
    
    # Anti-diagonal symmetry - only for square grids
    if row_count == column_count:
        symmetries[4] = True
        for i in range(row_count):
            for j in range(column_count):
                if state[i, j] != state[column_count - 1 - j, row_count - 1 - i]:
                    symmetries[4] = False
                    break
            if not symmetries[4]:
                break
    
    return symmetries

@njit(cache=True, nogil=True)
def filter_symmetric_actions_nb(valid_moves, state, row_count, column_count):
    """
    Filter valid moves to remove symmetric equivalents, keeping only canonical representatives.
    
    Args:
        valid_moves (np.ndarray): 1D boolean array of valid moves
        state (np.ndarray): 2D state array
        row_count (int): Number of rows
        column_count (int): Number of columns
    
    Returns:
        np.ndarray: Filtered valid moves array
    """
    # Detect current state symmetries
    symmetries = detect_state_symmetries_nb(state)
    
    # If no symmetries, return original valid moves
    if not np.any(symmetries):
        return valid_moves
    
    # Create a copy of valid moves to modify
    filtered_moves = valid_moves.copy()
    
    # Get list of valid action indices
    valid_indices = np.where(valid_moves)[0]
    
    # For each valid action, check if it should be kept as canonical representative
    for i, action in enumerate(valid_indices):
        if not filtered_moves[action]:  # Already filtered out
            continue
            
        # Generate all symmetric equivalents of this action
        symmetric_actions = np.empty(8, dtype=np.int64)
        symmetric_actions[0] = action  # Identity
        
        # Apply transformations based on detected symmetries
        transform_count = 1
        
        # Horizontal symmetry
        if symmetries[0]:
            symmetric_actions[transform_count] = apply_symmetry_transform_nb(
                action, 1, row_count, column_count)
            transform_count += 1
        
        # Vertical symmetry
        if symmetries[1]:
            symmetric_actions[transform_count] = apply_symmetry_transform_nb(
                action, 2, row_count, column_count)
            transform_count += 1
        
        # Central symmetry
        if symmetries[2]:
            symmetric_actions[transform_count] = apply_symmetry_transform_nb(
                action, 3, row_count, column_count)
            transform_count += 1
        
        # Main diagonal (only for square grids)
        if row_count == column_count and symmetries[3]:
            symmetric_actions[transform_count] = apply_symmetry_transform_nb(
                action, 4, row_count, column_count)
            transform_count += 1
        
        # Anti-diagonal (only for square grids)
        if row_count == column_count and symmetries[4]:
            symmetric_actions[transform_count] = apply_symmetry_transform_nb(
                action, 5, row_count, column_count)
            transform_count += 1
        
        # Find the canonical (smallest) action among equivalents
        canonical_action = action
        for t in range(transform_count):
            sym_action = symmetric_actions[t]
            if sym_action < canonical_action:
                canonical_action = sym_action
        
        # Keep only the canonical action, remove others
        for t in range(transform_count):
            sym_action = symmetric_actions[t]
            if sym_action != canonical_action and sym_action < len(filtered_moves):
                filtered_moves[sym_action] = False
    
    return filtered_moves

class N3il_with_symmetry(N3il):
    """
    Enhanced N3il class with symmetry detection capabilities.
    Inherits all functionality from N3il and adds symmetry analysis.
    Filters action space based on symmetries to improve performance.
    """
    
    def __init__(self, grid_size, args, priority_grid=None):
        # Initialize the parent N3il class
        super().__init__(grid_size, args, priority_grid)
    
    def get_valid_moves(self, state):
        """
        Override parent method to include symmetry filtering.
        
        Args:
            state (np.ndarray): Current state
            
        Returns:
            np.ndarray: Filtered valid moves considering symmetries
        """
        # Get original valid moves from parent
        valid_moves = super().get_valid_moves(state)
        
        # Apply symmetry filtering
        filtered_moves = filter_symmetric_actions_nb(
            valid_moves, state, self.row_count, self.column_count)
        
        return filtered_moves
    
    def get_valid_moves_subset(self, parent_state, parent_valid_moves, action_taken):
        """
        Override parent method to include symmetry filtering.
        
        Args:
            parent_state (np.ndarray): Parent state
            parent_valid_moves (np.ndarray): Parent's valid moves
            action_taken (int): Action that was taken
            
        Returns:
            np.ndarray: Filtered valid moves considering symmetries
        """
        # Get original valid moves from parent
        valid_moves = super().get_valid_moves_subset(parent_state, parent_valid_moves, action_taken)
        
        # Create child state for symmetry analysis
        child_state = parent_state.copy()
        row = action_taken // self.column_count
        col = action_taken % self.column_count
        child_state[row, col] = 1
        
        # Apply symmetry filtering
        filtered_moves = filter_symmetric_actions_nb(
            valid_moves, child_state, self.row_count, self.column_count)
        
        return filtered_moves


# Test the new class with symmetry filtering
print("=== Testing N3il_with_symmetry Class with Action Filtering ===")
print()

# Create an instance with symmetry detection and filtering
n3il_with_sym = N3il_with_symmetry((n, n), args=args)

# Test with current state
print("Current state:")
print(state)
print()

# Get valid moves without and with symmetry filtering for comparison
original_valid = super(N3il_with_symmetry, n3il_with_sym).get_valid_moves(state)
filtered_valid = n3il_with_sym.get_valid_moves(state)

print(f"Original valid moves count: {np.sum(original_valid)}")
print(f"Filtered valid moves count: {np.sum(filtered_valid)}")
print(f"Actions filtered out: {np.sum(original_valid) - np.sum(filtered_valid)}")
print()

# Test with a symmetric state (center point only)
symmetric_state = np.zeros((5, 5), dtype=np.uint8)
symmetric_state[2, 2] = 1  # Center point

print("=== Testing with Symmetric State (Center Point) ===")
print("Symmetric state:")
print(symmetric_state)
print()

original_valid_sym = super(N3il_with_symmetry, n3il_with_sym).get_valid_moves(symmetric_state)
filtered_valid_sym = n3il_with_sym.get_valid_moves(symmetric_state)

print(f"Original valid moves count: {np.sum(original_valid_sym)}")
print(f"Filtered valid moves count: {np.sum(filtered_valid_sym)}")
print(f"Actions filtered out: {np.sum(original_valid_sym) - np.sum(filtered_valid_sym)}")
print()

print("✅ N3il_with_symmetry class with action filtering successfully implemented!")
print("🔍 Features: Symmetry detection + action space reduction")
print("⚡ Performance: Numba-accelerated filtering for speed")
print("🎯 Perfect compatibility: Same interface as N3il for controlled comparison")

=== Testing N3il_with_symmetry Class with Action Filtering ===

Current state:
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Original valid moves count: 24
Filtered valid moves count: 14
Actions filtered out: 10

=== Testing with Symmetric State (Center Point) ===
Symmetric state:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Original valid moves count: 24
Filtered valid moves count: 5
Actions filtered out: 19

✅ N3il_with_symmetry class with action filtering successfully implemented!
🔍 Features: Symmetry detection + action space reduction
⚡ Performance: Numba-accelerated filtering for speed
🎯 Perfect compatibility: Same interface as N3il for controlled comparison
Original valid moves count: 24
Filtered valid moves count: 14
Actions filtered out: 10

=== Testing with Symmetric State (Center Point) ===
Symmetric state:
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Original valid moves count: 24
Filtered valid moves count: 5


In [20]:
state

array([[1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]], dtype=uint8)

In [19]:
n3il_with_sym.get_valid_moves(state).reshape((5,5))

array([[0, 1, 1, 1, 1],
       [0, 1, 1, 1, 1],
       [0, 0, 1, 1, 1],
       [0, 0, 0, 1, 1],
       [0, 0, 0, 0, 1]], dtype=uint8)

In [21]:
# Define all symmetric types of the current state
def get_all_symmetries(state):
    syms = []
    syms.append(state)  # original
    syms.append(np.flipud(state))  # horizontal
    syms.append(np.fliplr(state))  # vertical
    syms.append(np.flip(state, (0, 1)))  # central (180 rotation)
    if state.shape[0] == state.shape[1]:
        syms.append(state.T)  # main diagonal
        syms.append(np.flip(state.T, (0, 1)))  # anti-diagonal
    return syms

sym_names = [
    'original',
    'horizontal',
    'vertical',
    'central',
    'main_diagonal',
    'anti_diagonal'
]

all_syms = get_all_symmetries(state)
for idx, sym_state in enumerate(all_syms):
    print(f"\nSymmetry: {sym_names[idx]}")
    print(sym_state)
    print("Valid moves after symmetry filtering:")
    print(n3il_with_sym.get_valid_moves(sym_state).reshape((5,5)))


Symmetry: original
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[0 1 1 1 1]
 [0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 1 1]
 [0 0 0 0 1]]

Symmetry: horizontal
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 0]]

Symmetry: vertical
[[0 0 0 0 1]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 0]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [1 0 0 0 0]]

Symmetry: central
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 1]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [0 1 1 1 1]
 [0 0 1 1 1]
 [0 0 0 1 1]
 [0 0 0 0 0]]

Symmetry: main_diagonal
[[1 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
Valid moves after symmetry filtering:
[[1 1 1 1 1]
 [1 1 1 1 0]
 [1 1 1 0 0]
 [1 1 0 0 0]
 [0 0 0 0 0]]

Symmetry: vertical
[[0 0 0 0 1]
 [0 0 0 0 0

In [22]:
import numpy as np

# Define 8 distinct 5x5 states, each representing a D4 symmetry type
def make_example_states():
    states = []
    # 1. Identity (original)
    s0 = np.zeros((5,5), dtype=np.uint8)
    s0[1, 1] = 1; s0[2, 2] = 1; s0[3, 3] = 1
    states.append(s0)

    # 2. Horizontal flip
    s1 = np.flipud(s0)
    states.append(s1)

    # 3. Vertical flip
    s2 = np.fliplr(s0)
    states.append(s2)

    # 4. 180-degree rotation (central)
    s3 = np.flip(s0, (0, 1))
    states.append(s3)

    # 5. Main diagonal (transpose)
    s4 = s0.T
    states.append(s4)

    # 6. Anti-diagonal
    s5 = np.flip(s0.T, (0, 1))
    states.append(s5)

    # 7. Rotate 90 degrees (clockwise)
    s6 = np.rot90(s0, 1)
    states.append(s6)

    # 8. Rotate 270 degrees (counterclockwise)
    s7 = np.rot90(s0, 3)
    states.append(s7)

    return states

sym_names = [
    'identity',
    'horizontal',
    'vertical',
    'central',
    'main_diagonal',
    'anti_diagonal',
    'rotate_90',
    'rotate_270'
]

example_states = make_example_states()

for idx, s in enumerate(example_states):
    print(f'\nSymmetry type: {sym_names[idx]}')
    print(s)
    print('Valid moves:')
    print(n3il_with_sym.get_valid_moves(s).reshape((5,5)))


Symmetry type: identity
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: horizontal
[[0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[1 1 1 1 0]
 [0 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: vertical
[[0 0 0 0 0]
 [0 0 0 1 0]
 [0 0 1 0 0]
 [0 1 0 0 0]
 [0 0 0 0 0]]
Valid moves:
[[1 1 1 1 0]
 [0 1 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: central
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: main_diagonal
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Symmetry type: anti_diagonal
[[0 0 0 0 0]
 [0 1 0 0 0]
 [0 0 1 0 0]
 [0 0 0 1 0]
 [0 0 0 0 0]]
Valid moves:
[[0 1 1 1 1]
 [0 0 1 1 0]
 [0 0 0 0